# AiMO Agents — Contributor Guide

This notebook walks through the full project from first principles: environment setup,
the RAG pipeline, the teacher agent graph, quiz generation, and where to extend things.

**Prerequisites:** Python 3.11+, the `.venv` activated, and `pip install -e .` already run.
See the README for the one-time setup steps.

---

## Contents

1. [Environment & imports](#1-environment--imports)
2. [Configuration](#2-configuration)
3. [RAG pipeline — chunking, embedding, retrieval](#3-rag-pipeline)
4. [The `retrieve_material` tool](#4-the-retrieve_material-tool)
5. [Teacher agent — graph & nodes](#5-teacher-agent--graph--nodes)
6. [Generating a quiz end-to-end](#6-generating-a-quiz-end-to-end)
7. [Post-processing quality layer](#7-post-processing-quality-layer)
8. [Adding new study material](#8-adding-new-study-material)
9. [Running the tests](#9-running-the-tests)
10. [Where to extend the project](#10-where-to-extend-the-project)

## 1. Environment & imports

All public surface of the package is importable once `pip install -e .` has been run.
The cells below import only what is needed for each section so you can run them independently.

In [ ]:
import sys, pathlib

# Make sure the src/ package is on the path when running outside the installed venv
repo_root = pathlib.Path().resolve().parent
sys.path.insert(0, str(repo_root / "src"))

# Verify the package is importable
import aimo_agents
print("aimo_agents loaded from:", pathlib.Path(aimo_agents.__file__).parent)

## 2. Configuration

All runtime settings live in `src/aimo_agents/config/settings.py` and are loaded from
your `.env` file (or environment variables) via `load_settings()`.

No `.env` file? The defaults will kick in. Copy `.env.example` to `.env` to customise.

In [ ]:
from aimo_agents.config import load_settings

settings = load_settings()

print(f"Teacher model  : {settings.teacher_model_id}")
print(f"Material dir   : {settings.material_dir}")
print(f"RAG model      : {settings.rag_model_id}")
print(f"Chunk size     : {settings.rag_chunk_size} words  (overlap: {settings.rag_chunk_overlap})")
print(f"Top-k chunks   : {settings.rag_top_k}")
print(f"Score threshold: {settings.rag_score_threshold}")
print(f"Default Q count: {settings.default_question_count}")

## 3. RAG pipeline

The RAG module lives in `src/aimo_agents/rag/` and has three layers:

| Module | Class / function | Job |
|---|---|---|
| `chunker.py` | `chunk_text()` | Splits raw text into overlapping word windows |
| `embedder.py` | `Embedder` | Wraps `sentence-transformers` — returns L2-normalised float32 vectors |
| `retriever.py` | `TxtRetriever` | Indexes `.txt` files and retrieves top-k chunks by cosine similarity |

### 3a. Chunking

In [ ]:
from aimo_agents.rag.chunker import chunk_text

sample = "Colours are an important part of English vocabulary. " * 10
chunks = chunk_text(sample, chunk_size=20, overlap=5)

print(f"Input: {len(sample.split())} words  →  {len(chunks)} chunks")
for i, c in enumerate(chunks[:3], 1):
    print(f"\nChunk {i} ({len(c.split())} words):\n  {c[:80]}...")

### 3b. Retrieval with score threshold

`TxtRetriever.query()` accepts a `score_threshold` argument (cosine similarity, 0–1).
Any chunk below the threshold is discarded even if it ranks in the top-k.
This prevents off-topic chapters from leaking into the LLM context.

In [ ]:
import numpy as np
from aimo_agents.rag import TxtRetriever

# Build the index from the per-chapter files
retriever = TxtRetriever(
    model_id=settings.rag_model_id,
    chunk_size=settings.rag_chunk_size,
    overlap=settings.rag_chunk_overlap,
)
n = retriever.load_dir(settings.material_dir)
print(f"Indexed {n} chunks from {settings.material_dir}\n")

# Show top-8 scores for a colours query so the threshold effect is visible
query = "chapter colours colours"
q_emb = retriever._embedder.embed([query])
scores = (retriever._embeddings @ q_emb.T).squeeze()
top8 = np.argsort(scores)[::-1][:8].tolist()

print(f"{'Score':>7}  {'Status':6}  Preview")
print("-" * 70)
for i in top8:
    s = float(scores[i])
    status = "PASS" if s >= settings.rag_score_threshold else "FAIL"
    preview = retriever._chunks[i][:60].replace("\n", " ").strip()
    print(f"  {s:.3f}  [{status}]   {preview}")

---
## 4. The `retrieve_material` tool

`retrieve_material` is a LangChain `@tool` that wraps `TxtRetriever` behind a **lazy singleton** — the index is built once on first call and reused for every subsequent query.  The tool is invoked by the teacher agent's *retrieve* node and returns a single string of the concatenated top-k chunks that passed the score threshold.

```
topic input
     │
     ▼
 retrieve_material_tool.invoke({"chapter": "chapter <topic>", "topic": "<topic>"})
     │  (lazy-loads TxtRetriever on first call)
     ▼
 TxtRetriever.query(query, top_k, score_threshold)
     │
     └─► chunks that pass threshold ──► "\n\n".join(chunks)
```

In [ ]:
from aimo_agents.tools.retrieve_material import retrieve_material_tool

topic = "colours"
result = retrieve_material_tool.invoke({"chapter": f"chapter {topic}", "topic": topic})
print(f"Retrieved context ({len(result)} chars):\n")
print(result)

---
## 5. Teacher agent — graph & nodes

The teacher subgraph is a minimal two-node LangGraph `StateGraph`:

```
[START] ──► retrieve ──► generate ──► [END]
```

| Node | Function | File |
|---|---|---|
| `retrieve` | Calls `retrieve_material_tool`; stores result in `state["context"]` | `agents/teacher/nodes.py` |
| `generate` | Builds ChatML prompt → LLM → `_extract_json()` → `_postprocess()` | `agents/teacher/nodes.py` |

`build_teacher_graph(settings)` in `agents/teacher/graph.py` creates a HuggingFace text-generation pipeline, wraps it in a `HuggingFacePipeline`, and returns the compiled graph.

In [ ]:
from aimo_agents.agents.teacher.graph import build_teacher_graph

graph = build_teacher_graph(settings)

# LangGraph exposes the compiled graph structure
print("Nodes :", list(graph.nodes))
print("Edges :")
for edge in graph.edges:
    print(" ", edge)

---
## 6. Generating a quiz end-to-end

`run_once()` in `runner.py` is the main entry point. It:
1. Loads `settings` from `.env`
2. Builds the compiled agent graph
3. Invokes it with `{"topic": topic, "question_count": n, "messages": [HumanMessage(user_input)]}`
4. Saves the resulting quiz to `study_quiz.json` at the project root

Run the cell below to generate a fresh quiz (loads the Qwen model — takes ~30 s on first run).

In [ ]:
import json
from aimo_agents.runner import run_once

# Generate a 5-question quiz on "colours"
run_once(user_input="", topic="colours", question_count=5)

quiz_path = REPO_ROOT / "study_quiz.json"
quiz = json.loads(quiz_path.read_text(encoding="utf-8"))
print(json.dumps(quiz, indent=2))

---
## 7. Post-processing quality layer

Three filters run in sequence inside `_postprocess()` in `agents/teacher/nodes.py`:

| Filter | Purpose | Key parameters |
|---|---|---|
| Exact deduplication | Drops questions with identical text or identical option sets | — |
| `_leaks_answer()` | Drops a candidate if the correct answer of any accepted question appears verbatim in the candidate's question stem (or vice versa) | length > 2 guard |
| `_too_similar_to_any()` | Jaccard similarity on content words (question + correct answer); drops if ≥ threshold | `_OVERLAP_THRESHOLD = 0.40` |

A fourth function `_shuffle_answer()` randomises which letter slot holds the correct answer so the output isn't always "A".

The cells below demonstrate the leakage and similarity filters on hand-crafted examples.

In [ ]:
from aimo_agents.agents.teacher.nodes import _leaks_answer, _too_similar_to_any

# --- Leakage demo ---
# Q2 was accepted: correct answer = "a mix of red and blue"
accepted = [
    {
        "question": "What colour is made by mixing red and blue?",
        "options": {"A": "green", "B": "yellow", "C": "a mix of red and blue", "D": "orange"},
        "answer": "C",
    }
]

# Q3 candidate: its QUESTION BODY contains "a mix of red and blue" → should be flagged
candidate_leak = {
    "question": "Purple is a mix of red and blue. What colour is purple similar to?",
    "options": {"A": "purple", "B": "orange", "C": "yellow", "D": "pink"},
    "answer": "A",
}

# Q4 candidate: completely unrelated → should pass
candidate_ok = {
    "question": "What colour is the sky on a sunny day?",
    "options": {"A": "green", "B": "red", "C": "blue", "D": "orange"},
    "answer": "C",
}

print("Leakage test — candidate Q3 (should be LEAKED):",
      _leaks_answer(candidate_leak, accepted))
print("Leakage test — candidate Q4 (should be OK):    ",
      _leaks_answer(candidate_ok, accepted))

# --- Similarity demo ---
# Already-accepted question is identical in spirit to another candidate
accepted2 = [
    {
        "question": "What is the colour of grass?",
        "options": {"A": "blue", "B": "green", "C": "red", "D": "white"},
        "answer": "B",
    }
]
candidate_similar = {
    "question": "Grass has which colour?",
    "options": {"A": "yellow", "B": "purple", "C": "green", "D": "black"},
    "answer": "C",
}
print("\nSimilarity test — near-duplicate (should be TOO SIMILAR):",
      _too_similar_to_any(candidate_similar, accepted2))
print("Similarity test — candidate Q4 (should be OK):          ",
      _too_similar_to_any(candidate_ok, accepted2))

---
## 8. Adding new study material

To add a new topic or update an existing one:

1. **Create a chapter file** in `data/materials/chapters/`:
   ```
   data/materials/chapters/ch13_my_new_topic.txt
   ```
   Plain UTF-8 text.  Aim for 300–600 words — long enough to fill several chunks, short enough to stay focused.

2. **Keep it topic-focused** — the chapter file is the only piece the RAG sees for that topic, so don't mix topics within a single file.

3. **No code changes required** — `TxtRetriever.load_dir()` indexes every `.txt` file in the configured directory automatically.

4. **Verify retrieval quality** with the code from Section 3b — check that top-k scores are ≥ 0.30 for the intended query and that unrelated chapters score well below the threshold.

5. **Update `english_grade5.txt`** (the source-of-truth reference file) with the same content so the monolithic file stays in sync.

### Adjusting RAG hyperparameters

| Parameter | `.env` key | Effect |
|---|---|---|
| Context window size | `RAG_CHUNK_SIZE` | Larger = more context per chunk, harder to embed precisely |
| Overlap | `RAG_CHUNK_OVERLAP` | Higher = smoother boundary transitions, more chunks |
| Top-k results | `RAG_TOP_K` | More chunks = richer context, higher noise risk |
| Score filter | `RAG_SCORE_THRESHOLD` | Lower = more chunks pass, higher = stricter relevance |

---
## 9. Running the tests

The test suite lives in `tests/`.  Run it from the project root:

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", str(REPO_ROOT / "tests"), "-v", "--tb=short"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---
## 10. Where to extend the project

This section maps out the natural next steps for contributors.

### 10a. Orchestrator routing
`agents/orchestrator/` is currently a scaffold.  A working orchestrator would:
- Parse the user request to detect intent (quiz, explanation, revision, assessment)
- Route to the appropriate subgraph (`teacher`, or future `explainer`, `assessor`, …)
- Aggregate results and return a unified response

### 10b. Multi-subject corpora
The RAG is corpus-agnostic — any set of `.txt` files works.  To support multiple subjects:
- Group chapters by subject in sub-directories: `data/materials/english/`, `data/materials/maths/`
- Expose a `subject` parameter alongside `topic` in `retrieve_material_tool`
- Pass a per-subject `MATERIAL_DIR` through settings or a topics registry

### 10c. Difficulty levels
Add a `difficulty` field to `TeacherState` and pass it into the prompt.  Use few-shot examples with graduated difficulty (A1 vocabulary vs. B2 reasoning).

### 10d. Answer validation
Currently the post-processing filters are heuristic.  A stronger quality layer would:
- Re-embed each question + correct answer and check cosine similarity to the retrieved context
- Drop questions that have no strong semantic anchor in the source material

### 10e. Larger / quantised models
The pipeline is model-agnostic.  To try a larger model:
- Change `TEACHER_MODEL_ID` in `.env` (e.g. `Qwen/Qwen2.5-7B-Instruct-GGUF`)
- Set `DEVICE_MAP=auto` and ensure `bitsandbytes` is installed for 4-bit quantisation

### 10f. Streaming output
`HuggingFacePipeline` supports `streaming=True`.  Yield tokens through LangChain's streaming callback to give a real-time typing effect in a web UI.